# Clase 6 — Unidad 6: Visualización de Datos en Biomedicina

**Curso:** Introducción al Análisis de Datos — 2026, Segundo Semestre
**Material de referencia:** *Python Essentials for Biomedical Data Analysis* — Capítulo 6, "Data Visualization in Biomedicine"

## Objetivos de aprendizaje
- Comprender el **rol de la visualización** en la investigación biomédica.
- Conocer las principales **herramientas** de graficación en Python (Matplotlib, Seaborn).
- Crear los **gráficos básicos**: líneas, histogramas, dispersión, boxplots, mapas de calor de correlación, barras y tortas.
- Elegir **colores accesibles** y el tipo de gráfico adecuado a cada mensaje.
- Reflexionar sobre las **complejidades** (privacidad, claridad) al visualizar datos de salud.

## Contenidos de la clase
1. El rol de la visualización de datos en biomedicina
2. Herramientas de visualización en Python
3. Creación de gráficos básicos
4. Complejidades en la visualización de datos biomédicos
5. Visualización en la comunicación biomédica
6. Ejercicios de práctica

> Esta clase corresponde a la **Unidad 6** del libro guía. Trabajaremos con un conjunto de datos sintético de pacientes con **correlaciones realistas** inyectadas, para que los gráficos muestren patrones claros.

## Preparación del entorno

Instalamos **matplotlib** y **seaborn** (las dos librerías de graficación más usadas), además de pandas y numpy.

> **Sobre los colores:** usaremos la paleta **Okabe-Ito**, diseñada para ser distinguible por personas con daltonismo. Un buen gráfico elige el color según su función: un **tono único** para mostrar magnitud, una **paleta categórica** en orden fijo para distinguir grupos, y una escala **divergente** (dos colores con gris al centro) para valores que van de negativo a positivo, como las correlaciones.

In [ ]:
%pip install pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(6)
n = 200

# Generamos variables con correlaciones realistas
edad = np.random.randint(18, 90, n)
imc = np.random.normal(26, 4, n)
colesterol = np.random.normal(190, 30, n)
glucosa = 90 + 1.8 * (imc - 26) + 0.15 * (colesterol - 190) + np.random.normal(0, 8, n)
presion = 105 + 0.35 * edad + 0.6 * (imc - 26) + np.random.normal(0, 8, n)

df = pd.DataFrame({
    "edad": edad,
    "sexo": np.random.choice(["F", "M"], n),
    "imc": imc.round(1),
    "glucosa": glucosa.round(1),
    "colesterol": colesterol.round(1),
    "presion_sistolica": presion.round(0),
    "diagnostico": np.random.choice(["Sano", "Prediabetes", "Diabetes"], n, p=[0.55, 0.30, 0.15]),
})

# Paleta categórica accesible (Okabe-Ito) y color único para magnitudes
PALETA = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9"]
AZUL = "#0072B2"

print("Dataset creado:", df.shape)
df.head()

## 1. El rol de la visualización de datos en biomedicina

La visualización convierte datos complejos en representaciones visuales que facilitan **detectar patrones, tendencias y anomalías** que serían difíciles de ver en tablas de números. En biomedicina es clave para:
- Observar la **progresión de una enfermedad** o la respuesta a un tratamiento en el tiempo.
- Comparar **grupos de pacientes** (por diagnóstico, tratamiento, demografía).
- Explorar **relaciones** entre variables clínicas (ej. IMC y glucosa).
- **Comunicar resultados** a colegas, clínicos y público no experto.

Una buena visualización no solo *muestra* los datos: los hace **interpretables**.

## 2. Herramientas de visualización en Python

| Librería | Características |
|---|---|
| **Matplotlib** | La librería base. Control total y detallado sobre cada elemento del gráfico. |
| **Seaborn** | Construida sobre Matplotlib; gráficos estadísticos elegantes con menos código (heatmaps, pairplots). |
| **Plotly / Bokeh** | Gráficos **interactivos** (zoom, hover), ideales para dashboards y páginas web. |

En esta clase usaremos **Matplotlib** y **Seaborn**, que cubren la mayoría de las necesidades del análisis biomédico.

## 3. Creación de gráficos básicos

### 3.1 Gráfico de líneas — tendencias en el tiempo

Los **gráficos de líneas** son ideales para mostrar cómo cambia una variable a lo largo del tiempo (ej. niveles de un marcador tumoral, signos vitales, respuesta a tratamiento). Aquí comparamos la evolución de un marcador en 3 pacientes durante 12 meses.

In [ ]:
np.random.seed(1)
meses = np.arange(1, 13)

paciente1 = 30 + np.random.normal(0, 2, 12).cumsum() * 0.3   # estable con fluctuación
paciente2 = 25 + 0.8 * meses + np.random.normal(0, 1.5, 12)  # tendencia ascendente
paciente3 = 40 - 1.0 * meses + np.random.normal(0, 1.5, 12)  # tendencia descendente

plt.figure(figsize=(7, 4))
plt.plot(meses, paciente1, marker="o", color=PALETA[0], label="Paciente 1")
plt.plot(meses, paciente2, marker="s", color=PALETA[1], label="Paciente 2")
plt.plot(meses, paciente3, marker="^", color=PALETA[2], label="Paciente 3")
plt.title("Nivel de marcador tumoral en el tiempo")
plt.xlabel("Mes")
plt.ylabel("Nivel del marcador")
plt.legend()          # con 2 o más series, la leyenda es obligatoria
plt.tight_layout()
plt.show()

### 3.2 Histograma — distribución de una variable

Los **histogramas** muestran cómo se distribuye una variable numérica: si es simétrica o sesgada, si tiene uno o varios picos, o si hay valores atípicos. Al ser una sola variable (magnitud), usamos **un solo color**.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["glucosa"], bins=25, color=AZUL, edgecolor="white", alpha=0.85)
plt.title("Distribución de la glucosa")
plt.xlabel("Glucosa (mg/dL)")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

### 3.3 Diagrama de dispersión — relación entre dos variables

Los **scatter plots** revelan la relación entre dos variables numéricas: si hay correlación (positiva/negativa), tendencias o valores atípicos. Como inyectamos una relación entre IMC y glucosa, debería verse una tendencia positiva.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["imc"], df["glucosa"], alpha=0.6, color=AZUL)
plt.title("Relación entre IMC y glucosa")
plt.xlabel("IMC (kg/m²)")
plt.ylabel("Glucosa (mg/dL)")
plt.tight_layout()
plt.show()

**Seaborn** facilita explorar **todas las relaciones a la vez** con `pairplot()`: una matriz de gráficos de dispersión para cada par de variables, con la distribución de cada una en la diagonal.

In [ ]:
import seaborn as sns

sns.pairplot(df[["edad", "imc", "glucosa", "colesterol"]])
plt.show()

### 3.4 Boxplot — comparar la distribución entre grupos

El **boxplot** (diagrama de caja) resume una distribución en cuartiles y resalta los outliers. Es ideal para **comparar grupos**. Aquí comparamos la glucosa según el diagnóstico.

In [ ]:
grupos = ["Sano", "Prediabetes", "Diabetes"]
datos_grupos = [df[df["diagnostico"] == g]["glucosa"] for g in grupos]

plt.figure(figsize=(7, 4))
plt.boxplot(datos_grupos)
plt.xticks([1, 2, 3], grupos)
plt.title("Glucosa según diagnóstico")
plt.ylabel("Glucosa (mg/dL)")
plt.tight_layout()
plt.show()

### 3.5 Matriz de correlación y mapa de calor (heatmap)

Un **mapa de calor** traduce una matriz de números en colores, facilitando ver patrones. Para una **matriz de correlación** (valores de −1 a 1) usamos una escala **divergente** centrada en 0: un color para correlaciones positivas, otro para negativas y gris/blanco para las nulas.

In [ ]:
import seaborn as sns

numericas = df[["edad", "imc", "glucosa", "colesterol", "presion_sistolica"]]
correlacion = numericas.corr()

plt.figure(figsize=(6, 5))
sns.heatmap(correlacion, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, center=0, square=True)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

La función `clustermap` de Seaborn agrupa (clustering jerárquico) las variables con patrones similares y agrega **dendrogramas** en los bordes. Muy usada en datos de expresión génica.

In [ ]:
sns.clustermap(correlacion, annot=True, fmt=".2f", cmap="RdBu_r",
               vmin=-1, vmax=1, center=0, figsize=(6, 6))
plt.show()

### 3.6 Gráfico de barras — comparar cantidades entre grupos

Los **gráficos de barras** comparan un valor (frecuencia, media, proporción) entre categorías. Aquí mostramos la **glucosa promedio** por diagnóstico, con **barras de error** que representan el error estándar de la media (EEM).

In [ ]:
medias = [df[df["diagnostico"] == g]["glucosa"].mean() for g in grupos]
errores = [df[df["diagnostico"] == g]["glucosa"].sem() for g in grupos]   # error estándar de la media

plt.figure(figsize=(7, 4))
plt.bar(grupos, medias, yerr=errores, color=AZUL, alpha=0.85, capsize=5)
plt.title("Glucosa promedio por diagnóstico (± EEM)")
plt.ylabel("Glucosa media (mg/dL)")
plt.tight_layout()
plt.show()

### 3.7 Gráfico de torta (pie)

Los **gráficos de torta** muestran proporciones de un total. Al ser categorías, usamos la paleta categórica.

> ⚠️ **Nota:** en el análisis científico los gráficos de torta se usan con cautela, porque es difícil comparar visualmente el tamaño de los sectores. Suelen ser mejores para **audiencias no técnicas** y **pocas categorías**. Un gráfico de barras casi siempre comunica mejor.

In [ ]:
conteo = df["diagnostico"].value_counts()

plt.figure(figsize=(5, 5))
plt.pie(conteo.values, labels=conteo.index, autopct="%1.1f%%",
        startangle=140, colors=PALETA[:len(conteo)])
plt.title("Proporción de diagnósticos")
plt.axis("equal")   # asegura que la torta sea circular
plt.tight_layout()
plt.show()

## 4. Complejidades en la visualización de datos biomédicos

Visualizar datos de salud implica desafíos particulares:

- **Manejo efectivo de los datos:** conjuntos grandes y heterogéneos (imágenes, genómica, clínicos) requieren simplificar sin distorsionar el mensaje.
- **Aceptabilidad de la visualización:** el gráfico debe ser claro, honesto y no inducir a conclusiones erróneas (ej. ejes truncados que exageran diferencias).
- **Confidencialidad de datos sensibles:** nunca mostrar información que identifique a un paciente. Al graficar, usar datos agregados o anonimizados.

## 5. Visualización en la comunicación biomédica

Un mismo dato puede necesitar **distintas visualizaciones según la audiencia**:
- **Para no expertos** (pacientes, público, autoridades): gráficos simples y directos (barras, tortas), con lenguaje claro y sin jerga.
- **Para documentación y publicación científica:** gráficos precisos, con títulos, ejes rotulados, unidades y leyendas; reproducibles.
- **Para discusiones clínicas:** visualizaciones que resalten lo relevante para la decisión (ej. la evolución de un biomarcador de un paciente).

**Buenas prácticas transversales:** siempre incluir **título, etiquetas de ejes y unidades**; usar **colores accesibles**; elegir el **tipo de gráfico** adecuado al mensaje; y evitar sobrecargar el gráfico.

## 6. Ejercicios de práctica

Usa el conjunto `df` (o crea tus propios datos) para resolver:

1. **Histograma:** grafica la distribución de `colesterol`. ¿Se ve simétrica o sesgada?
2. **Dispersión:** haz un scatter de `edad` (X) vs `presion_sistolica` (Y). ¿Qué relación observas?
3. **Boxplot:** compara la distribución del `imc` entre hombres y mujeres (`sexo`).
4. **Barras:** grafica el `colesterol` promedio por diagnóstico, con barras de error (EEM).
5. **Torta:** muestra la proporción de pacientes por `sexo`.
6. **Mapa de calor:** crea la matriz de correlación de `edad`, `imc`, `glucosa` y `presion_sistolica`, y visualízala con un heatmap divergente.
7. **Líneas (desafío):** simula el peso de un paciente durante 10 semanas y grafícalo como una línea con marcadores, con título y ejes rotulados.

### Preguntas para reflexionar
1. ¿Qué tipo de gráfico usarías para mostrar la evolución de un biomarcador en el tiempo? ¿Y para comparar grupos?
2. ¿Por qué conviene usar una paleta de colores accesible para daltonismo?
3. ¿Por qué una escala **divergente** es la adecuada para una matriz de correlación?
4. ¿Qué riesgos éticos existen al visualizar datos de pacientes?
5. ¿Por qué los gráficos de torta se desaconsejan en contextos científicos?

In [ ]:
# Espacio para resolver los ejercicios
